# Accuracy Assessment: LoD2 2025 vs. LoD1 2015 Residential Subtypes

This notebook evaluates how well the residential subtype classification in the 2025 LoD2 existing-building dataset matches the residential subtype labels in the 2015 LoD1 dataset.

Method:
- use only `building_class == 'residential'` from the 2025 existing buildings
- convert LoD2 `res_subclass` labels to LoD1 classes 1-4
- create centroids from the 2025 buildings
- spatially join centroids to the 2015 LoD1 residential buildings
- compare predicted class from LoD2 with reference class from LoD1

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)
sns.set_theme(style='whitegrid', context='talk')

LOD2_EXISTING_GPKG = Path(r'C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\LoD2\LoD2_2025_existing_thr_h11m.gpkg')
LOD1_2015_GPKG = Path(r'C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Input\LoD1_2015\LoD1_2015_residential_buildings.gpkg')
OUTPUT_DIR = Path(r'C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\LoD2\Accuracy_Assessment_Residential_Subtypes_thr_h11m')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_MAP = {
    'SFH-DB': 1,
    'SBD': 2,
    'TB': 3,
    'MFH-AB': 4,
}
CLASS_LABELS = {
    1: 'SFH-DB',
    2: 'SBD',
    3: 'TB',
    4: 'MFH-AB',
}
CLASS_ORDER = [1, 2, 3, 4]

print('Loading data...')
gdf_lod2 = gpd.read_file(LOD2_EXISTING_GPKG)
gdf_lod1 = gpd.read_file(LOD1_2015_GPKG)

print(f'  LoD2 existing buildings: {len(gdf_lod2):,}')
print(f'  LoD1 residential buildings: {len(gdf_lod1):,}')
print(f'  LoD2 columns: {list(gdf_lod2.columns)}')
print(f'  LoD1 columns: {list(gdf_lod1.columns)}')

required_lod2 = {'building_class', 'res_subclass', 'geometry'}
required_lod1 = {'Klassifikation', 'geometry'}
missing_lod2 = required_lod2.difference(gdf_lod2.columns)
missing_lod1 = required_lod1.difference(gdf_lod1.columns)
if missing_lod2:
    raise KeyError(f'Missing LoD2 columns: {sorted(missing_lod2)}')
if missing_lod1:
    raise KeyError(f'Missing LoD1 columns: {sorted(missing_lod1)}')

if gdf_lod2.crs != gdf_lod1.crs:
    print(f'Reprojecting LoD1 from {gdf_lod1.crs} to {gdf_lod2.crs}')
    gdf_lod1 = gdf_lod1.to_crs(gdf_lod2.crs)

gdf_lod2 = gdf_lod2[gdf_lod2['building_class'].eq('residential')].copy()
gdf_lod2 = gdf_lod2[gdf_lod2['res_subclass'].isin(CLASS_MAP)].copy()
gdf_lod2['pred_class'] = gdf_lod2['res_subclass'].map(CLASS_MAP).astype('Int64')
gdf_lod2['lod2_id'] = gdf_lod2.index.astype(str)
gdf_lod2['geometry'] = gdf_lod2.geometry.centroid

gdf_lod1 = gdf_lod1.copy()
gdf_lod1['true_class'] = pd.to_numeric(gdf_lod1['Klassifikation'], errors='coerce').astype('Int64')
gdf_lod1 = gdf_lod1[gdf_lod1['true_class'].isin(CLASS_ORDER)].copy()

print()
print(f'Filtered LoD2 residential existing buildings: {len(gdf_lod2):,}')
print(f'Filtered LoD1 reference buildings: {len(gdf_lod1):,}')

join_cols = ['geometry', 'true_class']
joined = gpd.sjoin(
    gdf_lod2[['lod2_id', 'geometry', 'res_subclass', 'pred_class']].copy(),
    gdf_lod1[join_cols].copy(),
    how='left',
    predicate='within',
)

joined = joined.rename(columns={'index_right': 'lod1_match_index'})
joined = joined.drop_duplicates(subset=['lod2_id'], keep='first').copy()
joined['matched'] = joined['true_class'].notna()
matched = joined[joined['matched']].copy()
unmatched_count = int((~joined['matched']).sum())

print(f'LoD2 buildings with LoD1 match: {len(matched):,}')
print(f'LoD2 buildings without LoD1 match: {unmatched_count:,}')

if matched.empty:
    raise ValueError('No centroid matches found between LoD2 and LoD1 datasets.')

y_true = matched['true_class'].astype(int)
y_pred = matched['pred_class'].astype(int)
overall_accuracy = accuracy_score(y_true, y_pred)
conf = confusion_matrix(y_true, y_pred, labels=CLASS_ORDER)
report_dict = classification_report(
    y_true,
    y_pred,
    labels=CLASS_ORDER,
    target_names=[CLASS_LABELS[c] for c in CLASS_ORDER],
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report_dict).transpose()

conf_df = pd.DataFrame(
    conf,
    index=[f'true_{CLASS_LABELS[c]}' for c in CLASS_ORDER],
    columns=[f'pred_{CLASS_LABELS[c]}' for c in CLASS_ORDER],
)

summary_df = pd.DataFrame([
    {'metric': 'lod2_residential_existing_total', 'value': len(gdf_lod2)},
    {'metric': 'lod1_reference_total', 'value': len(gdf_lod1)},
    {'metric': 'matched_buildings', 'value': len(matched)},
    {'metric': 'unmatched_buildings', 'value': unmatched_count},
    {'metric': 'match_rate_pct', 'value': round(len(matched) / len(gdf_lod2) * 100, 3)},
    {'metric': 'overall_accuracy', 'value': round(overall_accuracy, 6)},
])

summary_path = OUTPUT_DIR / 'lod2_vs_lod1_residential_subtypes_summary.csv'
report_path = OUTPUT_DIR / 'lod2_vs_lod1_residential_subtypes_classification_report.csv'
conf_path = OUTPUT_DIR / 'lod2_vs_lod1_residential_subtypes_confusion_matrix.csv'
matches_path = OUTPUT_DIR / 'lod2_vs_lod1_residential_subtypes_matches.csv'

summary_df.to_csv(summary_path, index=False, encoding='utf-8-sig')
report_df.to_csv(report_path, encoding='utf-8-sig')
conf_df.to_csv(conf_path, encoding='utf-8-sig')
matched[['lod2_id', 'res_subclass', 'pred_class', 'true_class']].to_csv(matches_path, index=False, encoding='utf-8-sig')

print('\n' + '=' * 72)
print('ACCURACY ASSESSMENT: LOD2 2025 vs LOD1 2015 RESIDENTIAL SUBTYPES')
print('=' * 72)
print(f'Overall accuracy: {overall_accuracy:.3%}')
print(f'Matched buildings: {len(matched):,} / {len(gdf_lod2):,} ({len(matched)/len(gdf_lod2):.1%})')
print('\nClass distribution in matched sample:')
print(pd.crosstab(y_true, y_pred, rownames=['true_class'], colnames=['pred_class']).to_string())
print('\nClassification report:')
print(
    classification_report(
        y_true,
        y_pred,
        labels=CLASS_ORDER,
        target_names=[CLASS_LABELS[c] for c in CLASS_ORDER],
        zero_division=0,
    )
)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(conf_df, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax)
ax.set_title('LoD2 2025 vs LoD1 2015 Residential Subtypes')
ax.set_xlabel('Predicted class from LoD2 2025')
ax.set_ylabel('Reference class from LoD1 2015')
plt.tight_layout()
fig_path = OUTPUT_DIR / 'lod2_vs_lod1_residential_subtypes_confusion_matrix.png'
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()

print(f'\nSaved: {summary_path.name}')
print(f'Saved: {report_path.name}')
print(f'Saved: {conf_path.name}')
print(f'Saved: {matches_path.name}')
print(f'Saved: {fig_path.name}')

In [ ]:
# Focused diagnostic for SFH-DB vs MFH-AB misclassifications
import numpy as np

THRESH_MFH_HEIGHT = 9.4
THRESH_MFH_AREA_MIN = 129.6
THRESH_MFH_AREA = 190.8
THRESH_MFH_AREA_LARGE = 268.0
THRESH_SFH_AREA = 190.8
RES_TYPES_GPKG = Path(r'C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\LoD2\LoD2_2025_residential_types.gpkg')

gdf_existing_full = gpd.read_file(LOD2_EXISTING_GPKG)
gdf_res_types = gpd.read_file(RES_TYPES_GPKG)

if gdf_existing_full.crs != gdf_lod1.crs:
    gdf_existing_full = gdf_existing_full.to_crs(gdf_lod1.crs)
if gdf_res_types.crs != gdf_lod1.crs:
    gdf_res_types = gdf_res_types.to_crs(gdf_lod1.crs)

gdf_existing_full = gdf_existing_full[gdf_existing_full['building_class'].eq('residential')].copy()
gdf_existing_full = gdf_existing_full[gdf_existing_full['res_subclass'].isin(CLASS_MAP)].copy()
gdf_existing_full['pred_class_2025'] = gdf_existing_full['res_subclass'].map(CLASS_MAP).astype('Int64')
gdf_existing_full['lod2_id'] = gdf_existing_full.index.astype(str)
gdf_existing_pts = gpd.GeoDataFrame(
    gdf_existing_full.drop(columns='geometry').copy(),
    geometry=gdf_existing_full.geometry.centroid,
    crs=gdf_existing_full.crs,
 )

rule_cols = ['geometry', 'area_m2', 'measured_height', 'units_in_group', 'Rel_Area', 'res_subclass']
diag = gpd.sjoin(
    gdf_existing_pts,
    gdf_res_types[rule_cols].copy(),
    how='left',
    predicate='within',
)
diag = diag.drop(columns=['index_right'], errors='ignore')
diag = diag.drop_duplicates(subset=['lod2_id'], keep='first').copy()

diag = gpd.sjoin(
    diag,
    gdf_lod1[['geometry', 'true_class']].copy(),
    how='left',
    predicate='within',
)
diag = diag.drop(columns=['index_right'], errors='ignore')
diag = diag.drop_duplicates(subset=['lod2_id'], keep='first').copy()
diag = diag[diag['true_class'].notna()].copy()

def first_existing_column(frame, candidates):
    for candidate in candidates:
        if candidate in frame.columns:
            return candidate
    raise KeyError(f'None of these columns were found: {candidates}')

diag['area_m2'] = pd.to_numeric(diag[first_existing_column(diag, ['area_m2_right', 'area_m2'])], errors='coerce')
diag['measured_height'] = pd.to_numeric(diag[first_existing_column(diag, ['measured_height_right', 'measured_height'])], errors='coerce')
diag['units_in_group'] = pd.to_numeric(diag[first_existing_column(diag, ['units_in_group_right', 'units_in_group'])], errors='coerce')
diag['Rel_Area'] = pd.to_numeric(diag[first_existing_column(diag, ['Rel_Area_right', 'Rel_Area'])], errors='coerce')

diag['true_class'] = diag['true_class'].astype(int)
diag['pred_class_2025'] = diag['pred_class_2025'].astype(int)

sfh_to_mfh = diag[(diag['true_class'] == 1) & (diag['pred_class_2025'] == 4)].copy()
sfh_to_sfh = diag[(diag['true_class'] == 1) & (diag['pred_class_2025'] == 1)].copy()
mfh_to_sfh = diag[(diag['true_class'] == 4) & (diag['pred_class_2025'] == 1)].copy()
mfh_to_mfh = diag[(diag['true_class'] == 4) & (diag['pred_class_2025'] == 4)].copy()

def summarize_group(df, name):
    return {
        'group': name,
        'n': len(df),
        'area_median': round(df['area_m2'].median(), 2),
        'area_q75': round(df['area_m2'].quantile(0.75), 2),
        'area_q90': round(df['area_m2'].quantile(0.90), 2),
        'height_median': round(df['measured_height'].median(), 2),
        'height_q75': round(df['measured_height'].quantile(0.75), 2),
        'height_q90': round(df['measured_height'].quantile(0.90), 2),
        'units_eq_1_pct': round(df['units_in_group'].eq(1).mean() * 100, 2),
        'area_gt_190_8_pct': round(df['area_m2'].gt(THRESH_MFH_AREA).mean() * 100, 2),
        'area_gt_268_pct': round(df['area_m2'].gt(THRESH_MFH_AREA_LARGE).mean() * 100, 2),
        'height_area_rule_pct': round(((df['measured_height'] >= THRESH_MFH_HEIGHT) & (df['area_m2'] >= THRESH_MFH_AREA_MIN)).mean() * 100, 2),
    }

diagnostic_summary = pd.DataFrame([
    summarize_group(sfh_to_mfh, 'true SFH-DB -> pred MFH-AB'),
    summarize_group(sfh_to_sfh, 'true SFH-DB -> pred SFH-DB'),
    summarize_group(mfh_to_sfh, 'true MFH-AB -> pred SFH-DB'),
    summarize_group(mfh_to_mfh, 'true MFH-AB -> pred MFH-AB'),
])

rule_breakdown = pd.DataFrame([
    {'metric': 'true_SFH_pred_MFH_count', 'value': len(sfh_to_mfh)},
    {'metric': 'trigger_area_gt_268', 'value': int(sfh_to_mfh['area_m2'].gt(THRESH_MFH_AREA_LARGE).sum())},
    {'metric': 'trigger_area_gt_190_8', 'value': int(sfh_to_mfh['area_m2'].gt(THRESH_MFH_AREA).sum())},
    {'metric': 'trigger_height_area_rule', 'value': int(((sfh_to_mfh['measured_height'] >= THRESH_MFH_HEIGHT) & (sfh_to_mfh['area_m2'] >= THRESH_MFH_AREA_MIN)).sum())},
    {'metric': 'units_eq_1', 'value': int(sfh_to_mfh['units_in_group'].eq(1).sum())},
])

sfh_mfh_base = diag[diag['true_class'].isin([1, 4])].copy()

def simulate_rule_variant(area_threshold, height_threshold):
    pred_mfh = (
        sfh_mfh_base['area_m2'].gt(THRESH_MFH_AREA_LARGE)
        | sfh_mfh_base['area_m2'].gt(area_threshold)
        | ((sfh_mfh_base['measured_height'] >= height_threshold) & (sfh_mfh_base['area_m2'] >= THRESH_MFH_AREA_MIN))
    )
    pred_sfh = (
        sfh_mfh_base['units_in_group'].eq(1)
        & (
            (sfh_mfh_base['area_m2'] < THRESH_MFH_AREA_MIN)
            | (
                (sfh_mfh_base['area_m2'] <= THRESH_SFH_AREA)
                & (sfh_mfh_base['measured_height'].isna() | (sfh_mfh_base['measured_height'] < height_threshold))
            )
        )
    )
    pred_variant = np.where(pred_mfh, 4, np.where(pred_sfh, 1, sfh_mfh_base['pred_class_2025']))
    eval_df = pd.DataFrame({'true': sfh_mfh_base['true_class'], 'pred_variant': pred_variant})
    sfh_recall = (eval_df.loc[eval_df['true'] == 1, 'pred_variant'] == 1).mean()
    mfh_recall = (eval_df.loc[eval_df['true'] == 4, 'pred_variant'] == 4).mean()
    return {
        'mfh_area_threshold': area_threshold,
        'mfh_height_threshold': height_threshold,
        'sfh_recall': round(float(sfh_recall), 4),
        'mfh_recall': round(float(mfh_recall), 4),
        'balanced_recall': round(float((sfh_recall + mfh_recall) / 2), 4),
    }

threshold_scan = pd.DataFrame([
    simulate_rule_variant(area_threshold, height_threshold)
    for area_threshold in [190.8, 210.0, 230.0, 250.0, 268.0]
    for height_threshold in [9.4, 10.0, 10.5, 11.0]
]).sort_values('balanced_recall', ascending=False)

print('\n' + '=' * 72)
print('DIAGNOSTIC: SFH-DB vs MFH-AB MISCLASSIFICATIONS')
print('=' * 72)
print('\nFocused confusion block:')
print(
    pd.crosstab(
        diag['true_class'].map(CLASS_LABELS),
        diag['pred_class_2025'].map(CLASS_LABELS),
        rownames=['true_2015'],
        colnames=['pred_2025'],
    ).loc[['SFH-DB', 'MFH-AB'], ['SFH-DB', 'MFH-AB']].to_string()
 )

print('\nGroup summary:')
print(diagnostic_summary.to_string(index=False))

print('\nRule breakdown for true SFH-DB predicted as MFH-AB:')
print(rule_breakdown.to_string(index=False))

print('\nBest threshold variants (SFH-DB vs MFH-AB subset):')
print(threshold_scan.head(10).to_string(index=False))